In [2]:
# Complete Setup and Configuration
import json
import re
import logging
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from collections import Counter
import pandas as pd

# Configuration Settings
MODEL_ID = "meta-llama/Meta-Llama-3-8B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_QUANTIZATION = True

# File Paths
NER_OUTPUT_PATH = "extracted_entities_structured.json"
RELATIONS_OUTPUT_PATH = "extracted_relations_complete.json"
LABEL_STUDIO_OUTPUT = "label_studio_complete.json"
OPTIMIZED_RELATIONS_OUTPUT = "optimized_relations.json"
OPTIMIZED_LABEL_STUDIO_OUTPUT = "optimized_label_studio.json"

# Logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print("🔧 Configuration loaded successfully")
print(f"📊 Device: {DEVICE}")
print(f"📁 Input file: {NER_OUTPUT_PATH}")
print(f"💾 Output files will be created in current directory")

🔧 Configuration loaded successfully
📊 Device: cuda
📁 Input file: extracted_entities_structured.json
💾 Output files will be created in current directory


In [3]:
# Load NER Data and LLaMA Model
def load_ner_data(file_path):
    """Load and validate NER data from JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"✅ Loaded {len(data)} sentences from {file_path}")
        
        # Analyze entity types
        entity_types = {}
        for sentence in data[:10]:  # Sample first 10
            for entity in sentence['entities']:
                label = entity['label']
                entity_types[label] = entity_types.get(label, 0) + 1
        
        print(f"📊 Entity types found: {list(entity_types.keys())}")
        print(f"📋 Distribution: {entity_types}")
        return data
        
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")
        return []

def load_llama_model():
    """Load LLaMA 3 model and tokenizer."""
    global model, tokenizer
    
    if 'model' in globals() and model is not None:
        print("✅ Using existing LLaMA model")
        return model, tokenizer
    
    print("🔄 Loading LLaMA 3 model...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Configure quantization
    if USE_QUANTIZATION:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        print("🔧 Using 4-bit quantization")
    else:
        quantization_config = None
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=quantization_config,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    
    print("✅ LLaMA 3 model loaded successfully")
    return model, tokenizer

# Execute loading
sentences_with_entities = load_ner_data(NER_OUTPUT_PATH)
model, tokenizer = load_llama_model()

# Display sample data
if sentences_with_entities:
    print(f"\n📝 Sample sentence:")
    sample = sentences_with_entities[0]
    print(f"Text: {sample['original_sentence']}")
    print(f"Entities ({len(sample['entities'])}):")
    for entity in sample['entities']:
        print(f"  - {entity['label']}: '{entity['text']}' [pos {entity['start_char']}:{entity['end_char']}]")

✅ Loaded 1986 sentences from extracted_entities_structured.json
📊 Entity types found: ['CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 'COORDINATES']
📋 Distribution: {'CHANGE': 17, 'LOC': 7, 'LULC': 13, 'DATE': 7, 'PERCENT': 7, 'CARDINAL': 1, 'COORDINATES': 1}
🔄 Loading LLaMA 3 model...
🔧 Using 4-bit quantization


2025-06-10 11:13:38,711 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ LLaMA 3 model loaded successfully

📝 Sample sentence:
Text: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.
Entities (6):
  - CHANGE: 'results' [pos 11:18]
  - LOC: 'Thimphu' [pos 48:55]
  - LULC: 'city' [pos 56:60]
  - CHANGE: 'changed' [pos 65:72]
  - CHANGE: 'change' [pos 118:124]
  - DATE: '2050' [pos 161:165]


In [4]:
# Zero-Shot LLaMA Relationship Extraction Functions
def construct_zero_shot_prompt(sentence, entities):
    """Create zero-shot prompt for LLaMA relationship extraction."""
    # Group entities by type
    entities_by_type = {}
    for entity in entities:
        entity_type = entity['label']
        if entity_type not in entities_by_type:
            entities_by_type[entity_type] = []
        entities_by_type[entity_type].append(entity['text'])
    
    # Format entities
    entity_lines = []
    for entity_type, texts in entities_by_type.items():
        entity_lines.append(f"- {entity_type}: {', '.join(texts)}")
    entities_text = '\n'.join(entity_lines)
    
    prompt = f"""<task>
Analyze this LULC (Land Use Land Cover) sentence and extract semantic relationships between entities.
</task>

<sentence>
{sentence}
</sentence>

<entities>
{entities_text}
</entities>

<instructions>
Find meaningful relationships and output them as: ENTITY1 --RELATION--> ENTITY2

Focus on these relationship types:
- undergoes: LULC entities undergoing changes
- occurs_during: Changes happening at specific times
- located_in: Spatial relationships
- affects: Impact relationships
- causes: Causal relationships

Use exact entity text from above. If no relationships exist, output: NO_RELATIONS
</instructions>

<output>
"""
    return prompt

def extract_relations_with_llama(sentence, entities, model, tokenizer):
    """Extract relationships using LLaMA 3."""
    prompt = construct_zero_shot_prompt(sentence, entities)
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1500).to(model.device)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=150,
                temperature=0.3,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                num_return_sequences=1
            )
        
        # Extract generated text
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        prompt_text = tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)
        generated_text = full_output[len(prompt_text):].strip()
        
        # Clean output
        if "</output>" in generated_text:
            generated_text = generated_text.split("</output>")[0]
        
        # Filter meaningful lines
        lines = generated_text.strip().split('\n')
        relation_lines = []
        for line in lines[:8]:  # Max 8 relationships
            line = line.strip()
            if line and ('-->' in line or 'NO_RELATIONS' in line.upper()):
                relation_lines.append(line)
        
        return '\n'.join(relation_lines) if relation_lines else "NO_RELATIONS"
        
    except Exception as e:
        logging.error(f"LLaMA generation error: {e}")
        return "NO_RELATIONS"

def parse_llama_relations(output_text, entities):
    """Parse LLaMA output into structured relationships."""
    relations = []
    
    if "NO_RELATIONS" in output_text.upper():
        return relations
    
    lines = output_text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if '-->' not in line:
            continue
        
        try:
            # Parse: ENTITY1 --RELATION--> ENTITY2
            parts = line.split('-->')
            if len(parts) != 2:
                continue
                
            left_part = parts[0].strip()
            right_part = parts[1].strip()
            
            # Extract entity1 and relation
            if '--' in left_part:
                entity1_text, relation = left_part.rsplit('--', 1)
                entity1_text = entity1_text.strip()
                relation = relation.strip()
            else:
                continue
            
            entity2_text = right_part.strip()
            
            # Find entity indices
            entity1_idx = find_entity_index(entity1_text, entities)
            entity2_idx = find_entity_index(entity2_text, entities)
            
            if entity1_idx is not None and entity2_idx is not None and entity1_idx != entity2_idx:
                relations.append({
                    "from_entity_idx": entity1_idx,
                    "from_entity": entities[entity1_idx],
                    "to_entity_idx": entity2_idx,
                    "to_entity": entities[entity2_idx],
                    "relation_type": relation,
                    "confidence": "llama_zero_shot"
                })
                
        except Exception as e:
            logging.warning(f"Error parsing line '{line}': {e}")
            continue
    
    return relations

def find_entity_index(text, entities):
    """Find entity index by text matching."""
    text = text.lower().strip()
    
    # Exact match
    for i, entity in enumerate(entities):
        if text == entity['text'].lower():
            return i
    
    # Partial match
    for i, entity in enumerate(entities):
        entity_text = entity['text'].lower()
        if text in entity_text or entity_text in text:
            return i
    
    return None

print("✅ Zero-shot LLaMA extraction functions defined")

✅ Zero-shot LLaMA extraction functions defined


In [5]:
def create_fewshot_prompt(sentence, entities):
    """
    Create a few-shot prompt for relationship extraction.
    
    Args:
        sentence (str): The input sentence
        entities (list): List of entity dictionaries with 'text', 'label', etc.
    
    Returns:
        str: Formatted few-shot prompt
    """
    # Format entities for the prompt
    formatted_entities = []
    for entity in entities:
        formatted_entities.append(f"- {entity['label']}: {entity['text']}")
    
    formatted_entities_str = "\n".join(formatted_entities)
    
    # Open the prompt template file or use a multi-line string
    prompt_template = """<instructions>
You are an expert in Land Use Land Cover (LULC) relationship extraction. Your task is to identify relationships between entities in text about land use changes, urbanization, deforestation, and environmental change.

ENTITIES:
- LULC: Land types (forest, city, urban, agricultural, wetland, grassland, etc.)
- CHANGE: Change processes (increase, decrease, decline, expand, convert, transform)
- LOC: Locations (countries, cities, regions, areas)
- DATE: Temporal information (years, decades, periods)
- PERCENT: Numerical percentages or measurements
- PROCESS: Causative processes (urbanization, deforestation, climate change)

RELATIONSHIP TYPES (in priority order):
1. causes: Process/entity CAUSES a change (e.g., "urbanization causes forest decline")
2. undergoes: LULC entity undergoes a change (e.g., "forest undergoes decline")
3. occurs_during: Change happens during a time period (e.g., "decline occurs_during 2010-2020")
4. located_in: Entity is located somewhere (e.g., "forest located_in Amazon")
5. has_magnitude: Change has a specific magnitude (e.g., "decline has_magnitude 15%")
6. converts_to: One LULC transforms into another (e.g., "forest converts_to urban")
7. affects: One entity affects another (e.g., "deforestation affects biodiversity")

TASK:
1. Identify all entities in the text.
2. Extract all meaningful relationships between entities, focusing on CAUSATION.
3. Format each relationship as: Entity1 --relationship_type--> Entity2
4. Prioritize CAUSES relationships when possible.

Look at each example carefully before extracting relationships from the new text.
</instructions>

<examples>
EXAMPLE 1:
Sentence: "Urban expansion caused forest cover to decline by 15% between 2010 and 2020."
Entities:
- PROCESS: Urban expansion
- LULC: forest
- CHANGE: decline
- PERCENT: 15%
- DATE: 2010
- DATE: 2020

Relationships:
Urban expansion --causes--> decline
forest --undergoes--> decline
decline --has_magnitude--> 15%
decline --occurs_during--> 2010-2020

EXAMPLE 2:
Sentence: "Agricultural land was converted to residential areas in Beijing during 2015-2018."
Entities:
- LULC: Agricultural land
- CHANGE: converted
- LULC: residential areas
- LOC: Beijing
- DATE: 2015-2018

Relationships:
Agricultural land --undergoes--> converted
Agricultural land --converts_to--> residential areas
converted --located_in--> Beijing
converted --occurs_during--> 2015-2018

EXAMPLE 3:
Sentence: "Deforestation led to a 25% reduction in rainforest area in Brazil, contributing to climate change."
Entities:
- PROCESS: Deforestation
- PERCENT: 25%
- CHANGE: reduction
- LULC: rainforest
- LOC: Brazil
- PROCESS: climate change

Relationships:
Deforestation --causes--> reduction
rainforest --undergoes--> reduction
reduction --has_magnitude--> 25%
reduction --located_in--> Brazil
Deforestation --affects--> climate change

EXAMPLE 4:
Sentence: "Built-up area increased significantly from 52.88% to 65.5% in Thimphu city."
Entities:
- LULC: Built-up area
- CHANGE: increased
- PERCENT: 52.88%
- PERCENT: 65.5%
- LOC: Thimphu
- LULC: city

Relationships:
Built-up area --undergoes--> increased
increased --has_magnitude--> 52.88% to 65.5%
Built-up area --located_in--> Thimphu
city --located_in--> Thimphu

EXAMPLE 5:
Sentence: "Climate change resulted in wetland loss of 25% over the past decade."
Entities:
- PROCESS: Climate change
- LULC: wetland
- CHANGE: loss
- PERCENT: 25%
- DATE: past decade

Relationships:
Climate change --causes--> loss
wetland --undergoes--> loss
loss --has_magnitude--> 25%
loss --occurs_during--> past decade
</examples>

<target>
Sentence: "{input_text}"
Entities:
{formatted_entities}

Relationships:
"""
    
    # Format the prompt with the input sentence and entities
    formatted_prompt = prompt_template.format(
        input_text=sentence,
        formatted_entities=formatted_entities_str
    )
    
    return formatted_prompt

def extract_relationships_fewshot(sentence, entities, model, tokenizer):
    """
    Extract relationships using few-shot prompting.
    
    Args:
        sentence (str): Input sentence
        entities (list): List of entities
        model: The LLM model
        tokenizer: The tokenizer for the model
        
    Returns:
        list: Extracted relationships
    """
    # Create the few-shot prompt
    prompt = create_fewshot_prompt(sentence, entities)
    
    # Generate using the model with appropriate parameters
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=300,
            temperature=0.1,  # Lower temperature for consistency in few-shot
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )
    
    # Decode and extract the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the relationships part
    relationships_text = generated_text.split("<target>")[-1].split("Relationships:")[-1].strip()
    
    # Parse the relationships into structured format
    # (This can use your existing parsing function)
    parsed_relations = parse_fewshot_relations(relationships_text, entities)
    
    return parsed_relations

# Updated hybrid extraction to use few-shot approach
def hybrid_extraction_with_fewshot(sentence_data, use_fewshot=True, use_rules_fallback=True):
    """
    Enhanced hybrid extraction using few-shot learning as primary method.
    
    Args:
        sentence_data: Dictionary with sentence and entity information
        use_fewshot: Whether to use few-shot approach
        use_rules_fallback: Whether to fall back to rules if few-shot fails
        
    Returns:
        dict: Result with extracted relationships
    """
    sentence = sentence_data['original_sentence']
    entities = sentence_data['entities']
    
    result = {
        'sentence_id': sentence_data.get('sentence_id', 0),
        'article_id': sentence_data.get('article_id', ''),
        'original_sentence': sentence,
        'entities': entities,
        'fewshot_relations': [],
        'rule_relations': [],
        'final_relations': [],
        'fewshot_raw_output': '',
        'method_used': '',
        'error': None
    }
    
    if len(entities) < 2:
        result['error'] = 'Not enough entities for relationship extraction'
        result['method_used'] = 'skipped'
        return result
    
    # Try few-shot extraction first
    if use_fewshot:
        try:
            fewshot_raw_output = extract_relationships_fewshot(sentence, entities, model, tokenizer)
            result['fewshot_raw_output'] = fewshot_raw_output
            
            # Parse the few-shot output
            fewshot_relations = parse_fewshot_relations(fewshot_raw_output, entities)
            result['fewshot_relations'] = fewshot_relations
            
            if fewshot_relations:
                result['final_relations'] = fewshot_relations
                result['method_used'] = 'fewshot'
                return result
        except Exception as e:
            result['error'] = f"Few-shot error: {str(e)}"
    
    # Fall back to rules if few-shot failed or found no relations
    if use_rules_fallback:
        try:
            rule_relations = optimized_rule_based_relations(sentence, entities)
            result['rule_relations'] = rule_relations
            
            if rule_relations:
                result['final_relations'] = rule_relations
                result['method_used'] = 'rules_fallback'
                return result
        except Exception as e:
            result['error'] = f"Rules fallback error: {str(e)}"
    
    # If we get here, both methods failed or found no relations
    result['method_used'] = 'no_relations_found'
    return result

In [6]:
# Basic Rule-Based Relationship Extraction
def basic_rule_based_relations(sentence, entities):
    """Basic rule-based relationship extraction."""
    relations = []
    
    # Group entities by type
    by_type = {}
    for i, entity in enumerate(entities):
        entity_type = entity['label']
        if entity_type not in by_type:
            by_type[entity_type] = []
        by_type[entity_type].append((i, entity))
    
    # Rule 1: LULC undergoes CHANGE
    if 'LULC' in by_type and 'CHANGE' in by_type:
        for lulc_idx, lulc_entity in by_type['LULC']:
            for change_idx, change_entity in by_type['CHANGE']:
                relations.append({
                    "from_entity_idx": lulc_idx,
                    "from_entity": lulc_entity,
                    "to_entity_idx": change_idx,
                    "to_entity": change_entity,
                    "relation_type": "undergoes",
                    "confidence": "rule_basic"
                })
    
    # Rule 2: CHANGE occurs_during DATE
    if 'CHANGE' in by_type and 'DATE' in by_type:
        for change_idx, change_entity in by_type['CHANGE']:
            for date_idx, date_entity in by_type['DATE']:
                relations.append({
                    "from_entity_idx": change_idx,
                    "from_entity": change_entity,
                    "to_entity_idx": date_idx,
                    "to_entity": date_entity,
                    "relation_type": "occurs_during",
                    "confidence": "rule_basic"
                })
    
    # Rule 3: Spatial relationships with LOC
    if 'LOC' in by_type:
        for loc_idx, loc_entity in by_type['LOC']:
            # Connect to LULC
            if 'LULC' in by_type:
                for lulc_idx, lulc_entity in by_type['LULC']:
                    relations.append({
                        "from_entity_idx": lulc_idx,
                        "from_entity": lulc_entity,
                        "to_entity_idx": loc_idx,
                        "to_entity": loc_entity,
                        "relation_type": "located_in",
                        "confidence": "rule_basic"
                    })
            
            # Connect to CHANGE
            if 'CHANGE' in by_type:
                for change_idx, change_entity in by_type['CHANGE']:
                    relations.append({
                        "from_entity_idx": change_idx,
                        "from_entity": change_entity,
                        "to_entity_idx": loc_idx,
                        "to_entity": loc_entity,
                        "relation_type": "located_in",
                        "confidence": "rule_basic"
                    })
    
    return relations

print("✅ Basic rule-based extraction functions defined")

✅ Basic rule-based extraction functions defined


In [7]:
# Optimized Rule-Based Relationship Extraction
def optimized_rule_based_relations(sentence, entities):
    """Optimized rule-based extraction with smart filtering."""
    relations = []
    
    # Group entities by type with position info
    by_type = {}
    for i, entity in enumerate(entities):
        entity_type = entity['label']
        if entity_type not in by_type:
            by_type[entity_type] = []
        by_type[entity_type].append((i, entity))
    
    # Rule 1: LULC undergoes CHANGE (with proximity filtering)
    if 'LULC' in by_type and 'CHANGE' in by_type:
        for lulc_idx, lulc_entity in by_type['LULC']:
            closest_change = None
            min_distance = float('inf')
            
            # Find the closest meaningful CHANGE entity
            for change_idx, change_entity in by_type['CHANGE']:
                change_text = change_entity['text'].lower()
                # Skip non-meaningful changes
                if change_text in ['results', 'result']:
                    continue
                
                distance = abs(lulc_entity['start_char'] - change_entity['start_char'])
                if distance < min_distance and distance < 100:  # Within 100 characters
                    min_distance = distance
                    closest_change = (change_idx, change_entity)
            
            if closest_change:
                relations.append({
                    "from_entity_idx": lulc_idx,
                    "from_entity": lulc_entity,
                    "to_entity_idx": closest_change[0],
                    "to_entity": closest_change[1],
                    "relation_type": "undergoes",
                    "confidence": "rule_optimized",
                    "distance": min_distance
                })
    
    # Rule 2: Meaningful CHANGE occurs_during DATE
    if 'CHANGE' in by_type and 'DATE' in by_type:
        meaningful_changes = []
        for change_idx, change_entity in by_type['CHANGE']:
            change_text = change_entity['text'].lower()
            # Only include action-oriented changes
            if any(word in change_text for word in ['changed', 'change', 'increase', 'decrease', 'decline', 'grow', 'expand', 'convert']):
                meaningful_changes.append((change_idx, change_entity))
        
        for change_idx, change_entity in meaningful_changes:
            for date_idx, date_entity in by_type['DATE']:
                relations.append({
                    "from_entity_idx": change_idx,
                    "from_entity": change_entity,
                    "to_entity_idx": date_idx,
                    "to_entity": date_entity,
                    "relation_type": "occurs_during",
                    "confidence": "rule_optimized"
                })
    
    # Rule 3: Optimized spatial relationships
    if 'LOC' in by_type:
        for loc_idx, loc_entity in by_type['LOC']:
            # Connect primary LULC entities
            if 'LULC' in by_type:
                for lulc_idx, lulc_entity in by_type['LULC']:
                    relations.append({
                        "from_entity_idx": lulc_idx,
                        "from_entity": lulc_entity,
                        "to_entity_idx": loc_idx,
                        "to_entity": loc_entity,
                        "relation_type": "located_in",
                        "confidence": "rule_optimized"
                    })
            
            # Connect only main changes (avoid redundancy)
            if 'CHANGE' in by_type:
                main_changes = []
                for change_idx, change_entity in by_type['CHANGE']:
                    change_text = change_entity['text'].lower()
                    if change_text not in ['results', 'result'] and len(change_text) > 4:
                        main_changes.append((change_idx, change_entity))
                
                # Connect only the first main change
                if main_changes:
                    change_idx, change_entity = main_changes[0]
                    relations.append({
                        "from_entity_idx": change_idx,
                        "from_entity": change_entity,
                        "to_entity_idx": loc_idx,
                        "to_entity": loc_entity,
                        "relation_type": "located_in",
                        "confidence": "rule_optimized"
                    })
    
    # Rule 4: CHANGE affects MAGNITUDE (if PERCENT entities exist)
    if 'CHANGE' in by_type and 'PERCENT' in by_type:
        for change_idx, change_entity in by_type['CHANGE']:
            change_text = change_entity['text'].lower()
            if any(word in change_text for word in ['increase', 'decrease', 'decline', 'grow']):
                for percent_idx, percent_entity in by_type['PERCENT']:
                    relations.append({
                        "from_entity_idx": change_idx,
                        "from_entity": change_entity,
                        "to_entity_idx": percent_idx,
                        "to_entity": percent_entity,
                        "relation_type": "has_magnitude",
                        "confidence": "rule_optimized"
                    })
    
    return relations

print("✅ Optimized rule-based extraction functions defined")

✅ Optimized rule-based extraction functions defined


In [8]:
# Hybrid Processing Engine - Combines LLaMA + Rules
def hybrid_relationship_extraction(sentences_with_entities, use_llama=True, use_optimized_rules=False):
    """
    Complete hybrid system that tries LLaMA first, falls back to rules.
    """
    all_results = []
    
    print(f"🚀 HYBRID RELATIONSHIP EXTRACTION")
    print(f"📊 Processing {len(sentences_with_entities)} sentences")
    print(f"🤖 LLaMA enabled: {use_llama}")
    print(f"⚙️ Optimized rules enabled: {use_optimized_rules}")
    print("=" * 60)
    
    for idx, sentence_data in enumerate(tqdm(sentences_with_entities, desc="Processing")):
        sentence = sentence_data['original_sentence']
        entities = sentence_data['entities']
        article_id = sentence_data['article_id']
        
        result = {
            'sentence_id': idx,
            'article_id': article_id,
            'original_sentence': sentence,
            'entities': entities,
            'llama_relations': [],
            'basic_rule_relations': [],
            'optimized_rule_relations': [],
            'final_relations': [],
            'llama_raw_output': '',
            'method_used': '',
            'processing_details': {},
            'error': None
        }
        
        try:
            if len(entities) < 2:
                result['error'] = 'Insufficient entities (<2)'
                result['method_used'] = 'skipped'
                all_results.append(result)
                continue
            
            # Method 1: Try LLaMA first
            llama_success = False
            if use_llama:
                try:
                    raw_output = extract_relations_with_llama(sentence, entities, model, tokenizer)
                    result['llama_raw_output'] = raw_output
                    llama_relations = parse_llama_relations(raw_output, entities)
                    result['llama_relations'] = llama_relations
                    
                    if llama_relations:
                        result['final_relations'] = llama_relations
                        result['method_used'] = 'llama_zero_shot'
                        llama_success = True
                        
                except Exception as e:
                    logging.warning(f"LLaMA failed for sentence {idx}: {e}")
            
            # Method 2: Fallback to rules if LLaMA failed or disabled
            if not llama_success:
                if use_optimized_rules:
                    optimized_relations = optimized_rule_based_relations(sentence, entities)
                    result['optimized_rule_relations'] = optimized_relations
                    
                    if optimized_relations:
                        result['final_relations'] = optimized_relations
                        result['method_used'] = 'rule_optimized'
                    else:
                        # Last resort: basic rules
                        basic_relations = basic_rule_based_relations(sentence, entities)
                        result['basic_rule_relations'] = basic_relations
                        result['final_relations'] = basic_relations
                        result['method_used'] = 'rule_basic'
                else:
                    basic_relations = basic_rule_based_relations(sentence, entities)
                    result['basic_rule_relations'] = basic_relations
                    result['final_relations'] = basic_relations
                    result['method_used'] = 'rule_basic'
            
            # Store processing details
            result['processing_details'] = {
                'num_entities': len(entities),
                'num_final_relations': len(result['final_relations']),
                'entity_types': list(set([e['label'] for e in entities])),
                'relation_types': list(set([r['relation_type'] for r in result['final_relations']]))
            }
            
            # Show progress for first few sentences
            if idx < 5:
                print(f"\n📝 Sentence {idx} ({result['method_used']}): {len(result['final_relations'])} relations")
                for rel in result['final_relations'][:2]:  # Show first 2
                    from_ent = rel['from_entity']
                    to_ent = rel['to_entity']
                    print(f"   - {from_ent['text']} --{rel['relation_type']}--> {to_ent['text']}")
                    
        except Exception as e:
            result['error'] = str(e)
            result['method_used'] = 'failed'
            logging.error(f"Error processing sentence {idx}: {e}")
        
        all_results.append(result)
    
    return all_results

print("✅ Hybrid processing engine defined")

✅ Hybrid processing engine defined


In [9]:
import json
import uuid # For generating unique IDs if needed

def convert_annotations_to_predictions_format(input_filename="label_studio_complete.json", output_filename="label_studio_predictions_ALL_FIXED.json"):
    """
    Converts an existing Label Studio JSON file (with an 'annotations' array)
    to the 'predictions' format that correctly displays entities and relations.

    Args:
        input_filename (str): The name of the JSON file to load.
        output_filename (str): The name for the new JSON file with fixed predictions.
    """
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            tasks_data = json.load(f)
        if not isinstance(tasks_data, list):
            print(f"Error: Expected a list of tasks in {input_filename}, but got {type(tasks_data)}")
            return None
    except FileNotFoundError:
        print(f"Error: Input file '{input_filename}' not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{input_filename}'. Make sure it's a valid JSON file.")
        return None

    fixed_tasks_with_predictions = []
    task_counter = 0

    for task_idx, original_task in enumerate(tasks_data):
        task_counter += 1
        print(f"Processing Task {task_counter} (Original ID: {original_task.get('id', 'N/A')})...")

        new_task = {
            "data": original_task.get("data", {"text": "Error: Missing text data"}),
            "id": original_task.get("id", task_idx), # Preserve original task ID if available
            "predictions": []
        }

        prediction_result_items = []
        entity_id_map = {} # To map old entity IDs/indices to new unique prediction IDs

        # Assuming annotations are in the first element of the 'annotations' array
        # and results are in 'result' key of that annotation.
        annotations_array = original_task.get("annotations", [])
        if not annotations_array or not isinstance(annotations_array, list) or not annotations_array[0].get("result"):
            print(f"  Warning: Task {task_counter} has no valid 'annotations[0].result'. Skipping annotation conversion for this task.")
            fixed_tasks_with_predictions.append(new_task) # Add task even if no annotations to convert
            continue

        original_results = annotations_array[0].get("result", [])
        if not isinstance(original_results, list):
            print(f"  Warning: 'annotations[0].result' in Task {task_counter} is not a list. Skipping.")
            fixed_tasks_with_predictions.append(new_task)
            continue

        # --- Step 1: Process Entities and assign new IDs ---
        entity_counter = 0
        temp_relations_to_process = []

        for item_idx, original_item in enumerate(original_results):
            item_type = original_item.get("type")
            original_item_id = original_item.get("id", f"temp_id_{item_idx}") # Use temp ID if original doesn't exist

            if item_type == "labels": # This is an entity
                new_entity_id = f"ent_{task_idx}_{entity_counter}"
                entity_id_map[original_item_id] = new_entity_id # Map old ID to new ID

                prediction_result_items.append({
                    "value": original_item.get("value", {}),
                    "id": new_entity_id,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "score": original_item.get("score", 0.9) # Add a default score
                })
                entity_counter += 1
            elif item_type == "relation": # This is a relation, store it for later processing
                temp_relations_to_process.append(original_item)
            else:
                print(f"  Warning: Unknown item type '{item_type}' in Task {task_counter}. Item: {original_item}")


        # --- Step 2: Process Relations using the new entity IDs ---
        relation_counter = 0
        for original_relation in temp_relations_to_process:
            original_from_id = original_relation.get("value", {}).get("from")
            original_to_id = original_relation.get("value", {}).get("to")
            relation_label_type = original_relation.get("value", {}).get("type", "related_to") # Get relation type

            # Map old from/to IDs to new entity prediction IDs
            new_from_id = entity_id_map.get(original_from_id)
            new_to_id = entity_id_map.get(original_to_id)

            if new_from_id and new_to_id:
                prediction_result_items.append({
                    "from_id": new_from_id,
                    "to_id": new_to_id,
                    "type": "relation",
                    "direction": "right", # Default direction
                    "labels": [relation_label_type],
                    "from_name": "relation",
                    "to_name": "label",
                    "score": original_relation.get("score", 0.8) # Add a default score
                })
                relation_counter += 1
            else:
                print(f"  Warning: Could not map relation in Task {task_counter}. Original from/to: {original_from_id}/{original_to_id}. Missing one or both in entity_id_map.")
                if not new_from_id: print(f"    Original 'from' ID '{original_from_id}' not found in new entity IDs.")
                if not new_to_id: print(f"    Original 'to' ID '{original_to_id}' not found in new entity IDs.")


        new_task["predictions"].append({
            "model_version": "converted-v1.0",
            "score": 0.85, # Placeholder score for the overall prediction set
            "result": prediction_result_items
        })
        fixed_tasks_with_predictions.append(new_task)
        print(f"  Converted {entity_counter} entities and {relation_counter} relations.")


    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(fixed_tasks_with_predictions, f, indent=2, ensure_ascii=False)
        print(f"\n✅ Successfully converted and saved {len(fixed_tasks_with_predictions)} tasks to '{output_filename}'")
        print("You can now import this file into Label Studio using the 'predictions' format.")
    except Exception as e:
        print(f"\nError: Could not write to output file '{output_filename}': {e}")
        return None

    return fixed_tasks_with_predictions

# --- USAGE ---
# Replace 'your_input_file.json' with the actual name of your file
# that contains the tasks with the old 'annotations' structure.
# The output will be 'label_studio_predictions_ALL_FIXED.json'.

# Example: If your file is named 'label_studio_complete.json'
# input_file_to_fix = 'label_studio_complete.json'
# If you have a different file name, change it here:
input_file_to_fix = 'label_studio_predictions.json' # Or whatever your main JSON file is called

if input_file_to_fix:
    print(f"Attempting to convert '{input_file_to_fix}'...")
    fixed_data = convert_annotations_to_predictions_format(input_filename=input_file_to_fix)
    if fixed_data:
        print(f"Conversion complete. Check '{'label_studio_predictions_ALL_FIXED.json'}'")
    else:
        print("Conversion failed.")
else:
    print("Please specify the 'input_file_to_fix'.")


Attempting to convert 'label_studio_predictions.json'...
Processing Task 1 (Original ID: 1)...

✅ Successfully converted and saved 1 tasks to 'label_studio_predictions_ALL_FIXED.json'
You can now import this file into Label Studio using the 'predictions' format.
Conversion complete. Check 'label_studio_predictions_ALL_FIXED.json'


In [10]:
import json

# Ensure these global variables are defined from your setup cell:
# NER_OUTPUT_PATH, RELATIONS_OUTPUT_PATH, LABEL_STUDIO_OUTPUT

def save_results_and_create_label_studio_predictions(processing_results):
    """
    Save detailed results and create Label Studio import files in 'predictions' format.
    """
    
    # Use a new output path for the predictions file to distinguish it
    PREDICTIONS_OUTPUT_PATH = "label_studio_predictions_output.json"

    print("💾 SAVING RESULTS AND CREATING LABEL STUDIO FILES (PREDICTIONS FORMAT)")
    print("=" * 70)
    
    # Save detailed processing results (unchanged, just for your record)
    # This assumes RELATIONS_OUTPUT_PATH is defined globally.
    try:
        with open(RELATIONS_OUTPUT_PATH, 'w', encoding='utf-8') as f:
            json.dump(processing_results, f, indent=2, ensure_ascii=False)
        print(f"✅ Detailed results saved: {RELATIONS_OUTPUT_PATH} (for internal use)")
    except NameError:
        print("Warning: RELATIONS_OUTPUT_PATH not defined. Skipping detailed results save.")
    except Exception as e:
        print(f"Error saving detailed results: {e}")

    # Create Label Studio tasks in predictions format
    label_studio_tasks_for_predictions = []
    
    for result_idx, result in enumerate(processing_results):
        # We want to generate predictions even if no relations are found,
        # as the entities themselves are predictions.
        
        sentence = result['original_sentence']
        entities = result['entities']
        relations = result['final_relations']
        
        prediction_result_items = []
        entity_id_map = {} # Map our internal 'entity_id' to `value.id` for relations to reference

        # 1. Create entity predictions
        for i, entity in enumerate(entities):
            # Create a unique ID for each entity prediction within the task
            unique_entity_pred_id = f"ent_pred_{result_idx}_{i}" 
            entity_id_map[i] = unique_entity_pred_id # Map our internal index to this new ID

            entity_prediction = {
                "value": {
                    "start": entity['start_char'],
                    "end": entity['end_char'],
                    "text": entity['text'],
                    "labels": [entity['label']]
                },
                "id": unique_entity_pred_id, # Crucial: Unique ID for this specific prediction item
                "from_name": "label", # Matches <Labels name="label"> in Label Studio config
                "to_name": "text",    # Matches <Text name="text">
                "type": "labels",     # Type of prediction item
                "score": 0.95         # Example score for entity prediction
            }
            prediction_result_items.append(entity_prediction)
        
        # 2. Create relation predictions
        for j, relation in enumerate(relations):
            # Get the unique IDs of the source and target entities from the map
            source_entity_pred_id = entity_id_map.get(relation['from_entity_idx'])
            target_entity_pred_id = entity_id_map.get(relation['to_entity_idx'])

            # Only add relation if both source and target entity predictions were found
            if source_entity_pred_id and target_entity_pred_id:
                relation_prediction = {
                    "from_id": source_entity_pred_id, # References the ID of the source entity prediction
                    "to_id": target_entity_pred_id,   # References the ID of the target entity prediction
                    "type": "relation",               # Type of prediction item
                    "direction": "right",             # Default direction, can be 'left' or 'bi'
                    "labels": [relation['relation_type']], # Label for the relation (CRITICAL: an array of strings)
                    "from_name": "relation",          # Matches <Relations name="relation">
                    "to_name": "label",               # Relations link to labels (entities)
                    "score": 0.8                       # Example score for relation prediction
                }
                prediction_result_items.append(relation_prediction)
            else:
                print(f"Warning: Relation from_entity_idx={relation['from_entity_idx']} or to_entity_idx={relation['to_entity_idx']} not found for task {result['sentence_id']}. Skipping relation.")

        # Create the top-level 'predictions' object for the task
        task_predictions = {
            "model_version": "hybrid-model-v1.0", # Your model's version
            "score": 0.85, # Overall score for this prediction set
            "result": prediction_result_items # Combined list of entity and relation predictions
        }

        # Create the final Label Studio task in the predictions format
        formatted_task = {
            "data": {
                "text": sentence
            },
            # This is the key change: "predictions" instead of "annotations"
            "predictions": [task_predictions],
            "id": result['sentence_id'], # Keep original sentence ID
            "meta": { # Add metadata to the task for Label Studio
                "article_id": result['article_id'],
                "method_used": result['method_used'],
                "num_entities": len(entities),
                "num_relations": len(relations),
                "relation_types": list(set([r['relation_type'] for r in relations])),
                "confidence_scores": list(set([r.get('confidence', 'unknown') for r in relations]))
            }
        }
        
        label_studio_tasks_for_predictions.append(formatted_task)
    
    # Save the Label Studio predictions file
    # This assumes LABEL_STUDIO_OUTPUT is defined globally.
    try:
        with open(PREDICTIONS_OUTPUT_PATH, 'w', encoding='utf-8') as f:
            json.dump(label_studio_tasks_for_predictions, f, indent=2, ensure_ascii=False)
        print(f"✅ Label Studio predictions file saved: {PREDICTIONS_OUTPUT_PATH}")
        print(f"📊 Created {len(label_studio_tasks_for_predictions)} Label Studio tasks in predictions format")
    except NameError:
        print("Warning: PREDICTIONS_OUTPUT_PATH not defined. Skipping predictions file save.")
    except Exception as e:
        print(f"Error saving predictions file: {e}")

    return label_studio_tasks_for_predictions


# The create_comprehensive_statistics function can remain as is,
# as it analyzes the already processed 'processing_results' without
# needing to care about the Label Studio export format.
  
# --- Original create_comprehensive_statistics function (unchanged) ---
def create_comprehensive_statistics(processing_results):
    """Create comprehensive statistics and analysis."""
    
    print("\n📊 COMPREHENSIVE ANALYSIS")
    print("=" * 50)
    
    # Basic statistics
    total_sentences = len(processing_results)
    successful_extractions = sum(1 for r in processing_results if r['final_relations'])
    total_relations = sum(len(r['final_relations']) for r in processing_results)
    
    # Method distribution
    method_counts = {}
    for result in processing_results:
        method = result['method_used']
        method_counts[method] = method_counts.get(method, 0) + 1
    
    # Relationship type distribution
    relation_types = {}
    confidence_levels = {}
    
    for result in processing_results:
        for relation in result['final_relations']:
            rel_type = relation['relation_type']
            confidence = relation.get('confidence', 'unknown')
            
            relation_types[rel_type] = relation_types.get(rel_type, 0) + 1
            confidence_levels[confidence] = confidence_levels.get(confidence, 0) + 1
    
    # Print statistics
    print(f"📈 EXTRACTION STATISTICS:")
    print(f"- Total sentences processed: {total_sentences}")
    print(f"- Successful extractions: {successful_extractions}")
    print(f"- Total relationships extracted: {total_relations}")
    print(f"- Average relationships per sentence: {total_relations/total_sentences:.2f}")
    print(f"- Success rate: {successful_extractions/total_sentences*100:.1f}%")
    
    print(f"\n🔧 METHOD DISTRIBUTION:")
    for method, count in sorted(method_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"- {method}: {count} sentences ({count/total_sentences*100:.1f}%)")
    
    print(f"\n🔗 RELATIONSHIP TYPE DISTRIBUTION:")
    for rel_type, count in sorted(relation_types.items(), key=lambda x: x[1], reverse=True):
        print(f"- {rel_type}: {count} relationships")
    
    print(f"\n🎯 CONFIDENCE DISTRIBUTION:")
    for confidence, count in sorted(confidence_levels.items(), key=lambda x: x[1], reverse=True):
        print(f"- {confidence}: {count} relationships")
    
    # Sample high-quality relationships
    print(f"\n✨ SAMPLE HIGH-QUALITY RELATIONSHIPS:")
    sample_count = 0
    for result in processing_results[:10]:  # Check first 10
        if result['final_relations'] and sample_count < 5:
            for relation in result['final_relations'][:1]:  # One per sentence
                from_ent = relation['from_entity']
                to_ent = relation['to_entity']
                method = result['method_used']
                print(f"  {sample_count+1}. {from_ent['text']} --{relation['relation_type']}--> {to_ent['text']} ({method})")
                sample_count += 1
                if sample_count >= 5:
                    break
    
    # Save statistics
    statistics = {
        'extraction_summary': {
            'total_sentences': total_sentences,
            'successful_extractions': successful_extractions,
            'total_relations': total_relations,
            'success_rate': f"{successful_extractions/total_sentences*100:.1f}%",
            'avg_relations_per_sentence': round(total_relations/total_sentences, 2)
        },
        'method_distribution': method_counts,
        'relation_type_distribution': relation_types,
        'confidence_distribution': confidence_levels,
        'files_created': [] # Will be updated after this call if needed
    }
    
    # This part was missing a dynamic `LABEL_STUDIO_OUTPUT` definition
    # Assuming LABEL_STUDIO_OUTPUT is defined globally.
    try:
        statistics['files_created'].append(RELATIONS_OUTPUT_PATH)
        statistics['files_created'].append("label_studio_predictions_output.json") # New predictions file
    except NameError:
        print("Warning: Global path variables not defined for statistics update.")

    with open('extraction_statistics.json', 'w', encoding='utf-8') as f:
        json.dump(statistics, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Statistics saved: extraction_statistics.json")
    return statistics

# If you run the main pipeline, make sure it calls the new function:
# Example of how you would call it in your main pipeline execution cell:
# label_studio_tasks = save_results_and_create_label_studio_predictions(processing_results)  

In [11]:
# Test the Complete System on Your Example
def test_complete_system():
    """Test all approaches on the example sentence."""
    if not sentences_with_entities:
        print("❌ No test data available")
        return
    
    example = sentences_with_entities[0]
    sentence = example['original_sentence']
    entities = example['entities']
    
    print("🧪 TESTING COMPLETE SYSTEM ON EXAMPLE")
    print("=" * 60)
    print(f"📝 Sentence: {sentence}")
    print(f"🏷️ Entities ({len(entities)}):")
    for i, entity in enumerate(entities):
        print(f"  {i}: {entity['label']} = '{entity['text']}' [pos {entity['start_char']}:{entity['end_char']}]")
    
    print(f"\n🤖 METHOD 1: LLaMA Zero-Shot")
    try:
        llama_output = extract_relations_with_llama(sentence, entities, model, tokenizer)
        llama_relations = parse_llama_relations(llama_output, entities)
        print(f"Raw output: {llama_output}")
        print(f"Parsed relations: {len(llama_relations)}")
        for rel in llama_relations:
            from_ent = rel['from_entity']
            to_ent = rel['to_entity']
            print(f"  - {from_ent['text']} --{rel['relation_type']}--> {to_ent['text']}")
    except Exception as e:
        print(f"❌ LLaMA error: {e}")
        llama_relations = []
    
    print(f"\n⚙️ METHOD 2: Basic Rules")
    basic_relations = basic_rule_based_relations(sentence, entities)
    print(f"Basic rule relations: {len(basic_relations)}")
    for i, rel in enumerate(basic_relations):
        from_ent = rel['from_entity']
        to_ent = rel['to_entity']
        print(f"  {i+1}: {from_ent['text']} --{rel['relation_type']}--> {to_ent['text']}")
    
    print(f"\n🎯 METHOD 3: Optimized Rules") 
    optimized_relations = optimized_rule_based_relations(sentence, entities)
    print(f"Optimized rule relations: {len(optimized_relations)}")
    for i, rel in enumerate(optimized_relations):
        from_ent = rel['from_entity']
        to_ent = rel['to_entity']
        confidence = rel.get('confidence', 'unknown')
        print(f"  {i+1}: {from_ent['text']} --{rel['relation_type']}--> {to_ent['text']} ({confidence})")
    
    print(f"\n📊 COMPARISON:")
    print(f"- LLaMA relations: {len(llama_relations)}")
    print(f"- Basic rule relations: {len(basic_relations)}")
    print(f"- Optimized rule relations: {len(optimized_relations)}")
    
    return llama_relations

# Run the test
test_results = test_complete_system()

🧪 TESTING COMPLETE SYSTEM ON EXAMPLE
📝 Sentence: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.
🏷️ Entities (6):
  0: CHANGE = 'results' [pos 11:18]
  1: LOC = 'Thimphu' [pos 48:55]
  2: LULC = 'city' [pos 56:60]
  3: CHANGE = 'changed' [pos 65:72]
  4: CHANGE = 'change' [pos 118:124]
  5: DATE = '2050' [pos 161:165]

🤖 METHOD 1: LLaMA Zero-Shot


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Raw output: results --undergoes--> changed
Thimphu --located_in--> city
Thimphu --occurs_during--> 2050
Parsed relations: 3
  - results --undergoes--> changed
  - Thimphu --located_in--> city
  - Thimphu --occurs_during--> 2050

⚙️ METHOD 2: Basic Rules
Basic rule relations: 10
  1: city --undergoes--> results
  2: city --undergoes--> changed
  3: city --undergoes--> change
  4: results --occurs_during--> 2050
  5: changed --occurs_during--> 2050
  6: change --occurs_during--> 2050
  7: city --located_in--> Thimphu
  8: results --located_in--> Thimphu
  9: changed --located_in--> Thimphu
  10: change --located_in--> Thimphu

🎯 METHOD 3: Optimized Rules
Optimized rule relations: 5
  1: city --undergoes--> changed (rule_optimized)
  2: changed --occurs_during--> 2050 (rule_optimized)
  3: change --occurs_during--> 2050 (rule_optimized)
  4: city --located_in--> Thimphu (rule_optimized)
  5: changed --located_in--> Thimphu (rule_optimized)

📊 COMPARISON:
- LLaMA relations: 3
- Basic rule 

In [12]:
# Execute Complete Processing Pipeline
def run_complete_pipeline():
    """Run the complete relationship extraction pipeline."""
    
    if not sentences_with_entities:
        print("❌ No input data available. Please check your NER file.")
        return None, None, None
    
    print("🚀 RUNNING COMPLETE RELATIONSHIP EXTRACTION PIPELINE")
    print("=" * 60)
    
    # Execute hybrid processing
    print("Step 1: Running hybrid relationship extraction...")
    processing_results = hybrid_relationship_extraction(
        sentences_with_entities,
        use_llama=True,  
        use_optimized_rules=False
    )
    
    print("Step 2: Saving results and creating Label Studio files...")
    label_studio_tasks = save_results_and_create_label_studio_predictions(processing_results)
    
    print("Step 3: Creating comprehensive statistics...")
    statistics = create_comprehensive_statistics(processing_results)
    
    print(f"\n🎉 PIPELINE COMPLETED SUCCESSFULLY!")
    print(f"📁 Files created:")
    print(f"  ✅ {RELATIONS_OUTPUT_PATH} - Detailed extraction results")
    print(f"  ✅ {LABEL_STUDIO_OUTPUT} - Label Studio import file")
    print(f"  ✅ extraction_statistics.json - Comprehensive statistics")
    
    return processing_results, label_studio_tasks, statistics

# Execute the complete pipeline
print("Starting complete pipeline execution...")
results, label_studio_data, stats = run_complete_pipeline()

Starting complete pipeline execution...
🚀 RUNNING COMPLETE RELATIONSHIP EXTRACTION PIPELINE
Step 1: Running hybrid relationship extraction...
🚀 HYBRID RELATIONSHIP EXTRACTION
📊 Processing 1986 sentences
🤖 LLaMA enabled: True
⚙️ Optimized rules enabled: False


Processing:   0%|          | 0/1986 [00:00<?, ?it/s]


📝 Sentence 0 (rule_basic): 10 relations
   - city --undergoes--> results
   - city --undergoes--> changed

📝 Sentence 1 (rule_basic): 4 relations
   - increase --occurs_during--> 2002
   - increase --occurs_during--> 2018

📝 Sentence 2 (llama_zero_shot): 4 relations
   - forest --undergoes--> declined
   - declined --occurs_during--> 15.25%

📝 Sentence 3 (llama_zero_shot): 1 relations
   - changes --undergoes--> urban

📝 Sentence 4 (rule_basic): 0 relations
Step 2: Saving results and creating Label Studio files...
💾 SAVING RESULTS AND CREATING LABEL STUDIO FILES (PREDICTIONS FORMAT)
✅ Detailed results saved: extracted_relations_complete.json (for internal use)
✅ Label Studio predictions file saved: label_studio_predictions_output.json
📊 Created 1986 Label Studio tasks in predictions format
Step 3: Creating comprehensive statistics...

📊 COMPREHENSIVE ANALYSIS
📈 EXTRACTION STATISTICS:
- Total sentences processed: 1986
- Successful extractions: 1396
- Total relationships extracted: 7010